<a href="https://colab.research.google.com/github/soleildayana/Apophis-Asteroid-Project/blob/main/dart_inverso/nb00_parches_conicos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 00 — Modelo de Parches Cónicos
## Diseño de misión DART inverso: Tierra → Apophis (2029)

**Autor:** Soleil Dayana Niño Murcia — 1033097666  
**Curso:** Mecánica Celeste  
**Fecha:** Mayo 2026

---

> **Objetivo:** Diseñar una misión de interceptación al asteroide (99942) Apophis usando el modelo de
> **parches cónicos** (patched-conic approximation). Calculamos las ΔV de cada fase y construimos la
> curva **ΔV(τ)** que muestra el costo de propulsión como función del tiempo de lanzamiento antes del encuentro,
> identificando la ventana óptima de misión.

---

### Contexto: ¿Qué es el DART Inverso?

La misión **DART** (Double Asteroid Redirection Test, NASA 2022) impactó cinéticamente el asteroide Dimorphos
para demostrar la redirección orbital como técnica de defensa planetaria.

El **DART inverso** plantea el escenario análogo para Apophis: diseñar una misión que parta desde la Tierra
y alcance al asteroide **antes** de su acercamiento histórico del 13 de abril de 2029.  
En este caso la geometría está *invertida* respecto a DART: no elegimos cuándo llega el objetivo,
sino que optimizamos **cuándo salimos** desde la Tierra para llegar a tiempo con el menor costo de ΔV.

### Estructura del notebook

| Sección | Contenido |
|---------|----------|
| 1 | Teoría: modelo de parches cónicos (3 fases) |
| 2 | Teoría: ecuación de vis-viva y parámetros hiperbólicos |
| 3 | Teoría: problema de Lambert |
| 4 | Setup: imports, constantes, unidades canónicas |
| 5 | Efemérides: Apophis en llegada, Tierra en ventana de lanzamiento |
| 6 | Ejemplo de transferencia: τ = 300 días |
| 7 | Barrido ΔV(τ) sobre la ventana completa |
| 8 | Visualización y ventana óptima |

## 1. Teoría: Modelo de Parches Cónicos (3 fases)

La **aproximación de parches cónicos** divide una trayectoria interplanetaria en segmentos cónicos
(cada uno dominado por un único cuerpo) y los "emparchea" en los límites de las esferas de influencia.

```
LEO ──(ΔV₁)──▶  Hipérbola de escape  ──▶  Elipse de transferencia  ──▶  Hipérbola de llegada ──(ΔV₂)──▶ Apophis
               [Fase 1: geocéntrica]      [Fase 2: heliocéntrica]      [Fase 3: apofocéntrica]
```

### Fase 1 — Escape geocéntrico

La nave parte de una **órbita de estacionamiento circular** (LEO, altitud $h_{\rm LEO} = 200$ km)
y se le aplica el primer impulso $\Delta V_1$ en el perigeo de una **hipérbola de escape**.

El exceso de velocidad hiperbólico $v_\infty$ es la velocidad que la nave tendrá al cruzar la esfera
de influencia terrestre (lejos de la Tierra), y coincide con la diferencia entre la velocidad de la
nave al inicio de la arco de transferencia heliocéntrico y la velocidad orbital de la Tierra:

$$v_{\infty,1} = \|\mathbf{V}_{\rm trans}(t_1) - \mathbf{V}_{\oplus}(t_1)\|$$

Desde LEO, para inyectarse en esa hipérbola, se aplica un impulso:

$$\boxed{\Delta V_1 = \sqrt{v_{\infty,1}^2 + \frac{2\mu_\oplus}{r_p}} - \sqrt{\frac{\mu_\oplus}{r_p}}}$$

donde $r_p = R_\oplus + h_{\rm LEO}$ es el radio de la órbita de estacionamiento.

### Fase 2 — Transferencia heliocéntrica (arco de Lambert)

Una vez fuera de la esfera de influencia terrestre, la nave sigue una **cónica heliocéntrica**
(elipse o hipérbola) que conecta la posición de la Tierra en $t_1$ (lanzamiento) con la posición
de Apophis en $t_2$ (llegada). Este segmento lo resuelve el **problema de Lambert**:

> Dados dos vectores de posición $\mathbf{r}_1$, $\mathbf{r}_2$ y un tiempo de vuelo $\Delta t = t_2 - t_1$,
> encontrar la cónica kepleriana que une ambos puntos en ese tiempo.

La solución entrega las velocidades $\mathbf{V}_1^{(\rm trans)}$ y $\mathbf{V}_2^{(\rm trans)}$ en cada
extremo del arco.

### Fase 3 — Llegada a Apophis

Al llegar a la posición de Apophis, la nave tiene un exceso de velocidad relativo al asteroide:

$$v_{\infty,2} = \|\mathbf{V}_2^{(\rm trans)} - \mathbf{V}_{\rm Apophis}(t_2)\|$$

- **Misión impactadora** (DART-like): no se requiere frenado. La nave choca con $v_{\infty,2}$. **$\Delta V_2 = 0$.**
- **Misión de encuentro** (rendezvous): la nave debe igualar la velocidad de Apophis → **$\Delta V_2 \approx v_{\infty,2}$.**

### ΔV total

$$\boxed{\Delta V_{\rm total}(\tau) = \Delta V_1(\tau) + \Delta V_2(\tau)}$$

donde $\tau$ es el tiempo de vuelo (días entre lanzamiento y llegada). La curva $\Delta V_{\rm total}(\tau)$
define la **ventana de lanzamiento**.

## 2. Teoría: Ecuación de Vis-Viva y Parámetros Hiperbólicos

### 2.1 Vis-viva (general)

Para cualquier cónica kepleriana con semiejor mayor $a$ alrededor de un cuerpo con $\mu = GM$:

$$\boxed{v^2 = \mu\left(\frac{2}{r} - \frac{1}{a}\right)}$$

| Tipo de órbita | $a$ | $E_\text{específica}$ |
|---|---|---|
| Elipse | $a > 0$ | $-\mu/(2a) < 0$ |
| Parábola | $a \to \infty$ | $0$ |
| Hipérbola | $a < 0$ | $-\mu/(2a) > 0$ |

### 2.2 Velocidad de escape y exceso hiperbólico

La **velocidad de escape** desde un radio $r$ es:
$$v_{\rm esc}(r) = \sqrt{\frac{2\mu}{r}}$$

Para una hipérbola con $a < 0$, la energía específica es $\mathcal{E} = v_\infty^2 / 2 = -\mu/(2a)$,
donde $v_\infty$ es el **exceso de velocidad hiperbólico** (velocidad al infinito).

La velocidad en el **perigeo** de la hipérbola de escape (radio $r_p$) se obtiene directamente de vis-viva:

$$v_{p} = \sqrt{v_\infty^2 + \frac{2\mu}{r_p}} = \sqrt{v_\infty^2 + v_{\rm esc}^2(r_p)}$$

La velocidad circular en la órbita de estacionamiento (LEO) es:
$$v_{\rm circ} = \sqrt{\frac{\mu}{r_p}}$$

Por lo tanto el impulso de salida requerido es:
$$\Delta V_1 = v_p - v_{\rm circ} = \sqrt{v_\infty^2 + v_{\rm esc}^2(r_p)} - v_{\rm circ}$$

> **Nota:** Esta expresión es monotónica creciente en $v_\infty$: cuanto mayor el exceso hiperbólico
> requerido (órbita de transferencia más energética), mayor el impulso desde LEO.

## 3. Teoría: Problema de Lambert

### 3.1 Enunciado

Dados:
- $\mathbf{r}_1$: posición de salida (Tierra en $t_1$)
- $\mathbf{r}_2$: posición de llegada (Apophis en $t_2$)
- $\Delta t = t_2 - t_1$: tiempo de vuelo
- $\mu$: parámetro gravitacional del cuerpo central (Sol)

**Encontrar** las velocidades $\mathbf{V}_1$ y $\mathbf{V}_2$ de la cónica kepleriana que conecta
$\mathbf{r}_1$ con $\mathbf{r}_2$ en el tiempo $\Delta t$.

### 3.2 Dirección de la transferencia

Para cada par $(\mathbf{r}_1, \mathbf{r}_2, \Delta t)$ existen dos soluciones principales:

| Solución | Condición | Ángulo de transferencia |
|---|---|---|
| **Prograde** (`pro`) | $\Delta\nu < 180°$ | Órbita directa (sentido antihorario) |
| **Retrograde** (`retro`) | $\Delta\nu > 180°$ | Órbita retrógrada (sentido horario) |

En `pymcel` se invoca como:
```python
V1, V2, info = pc.solucion_lambert(r1, r2, tf, mu=1.0, direccion='pro')
```

### 3.3 Interpretación geométrica

La solución de Lambert determina completamente la **elipse de transferencia**: dado que conocemos
$\mathbf{r}_1$, $\mathbf{V}_1$ podemos calcular el semiejor mayor $a$, la excentricidad $e$
y todos los elementos orbitales mediante vis-viva y las relaciones del problema de dos cuerpos.

La ventana de lanzamiento emerge de barrer $t_1$ (o equivalentemente $\tau = \Delta t$)
con $t_2 = 2029$-$04$-$13$ fijo, y evaluar $\Delta V_{\rm total}(\tau)$ en cada punto.

In [ ]:
%pip install -Uq pymcel

## 4. Setup: imports, constantes y unidades canónicas

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

import pymcel as pc
from pymcel import constantes as const

print('Librerías cargadas correctamente.')

In [ ]:
# ── Unidades canónicas (sistema AU · M☉ · UT, G = 1) ─────────────────────────
AU_km    = 149_597_870.7          # km por AU
AU_m     = AU_km * 1e3            # m por AU
M_sun_kg = 1.989e30               # kg
G_SI     = 6.674e-11              # m³ kg⁻¹ s⁻²

UT_s     = np.sqrt(AU_m**3 / (G_SI * M_sun_kg))   # segundos
UT_days  = UT_s / 86_400.0                          # días
vel_unit = AU_km / UT_s * 1e-3                     # AU/UT → km/s  (factor de conversión)

mu_sun = 1.0   # unidades canónicas

# ── Parámetros terrestres (en km y km/s para la Fase 1) ─────────────────────
GM_earth_km3s2 = 398_600.4418     # km³/s²
R_earth_km     = 6_371.0          # km (radio medio)
h_leo_km       = 200.0            # km (altitud LEO de estacionamiento)
r_parking_km   = R_earth_km + h_leo_km   # km

v_circ_km_s  = np.sqrt(GM_earth_km3s2 / r_parking_km)   # km/s (vel. circular LEO)
v_esc_km_s   = np.sqrt(2 * GM_earth_km3s2 / r_parking_km)  # km/s (vel. de escape LEO)

print(f'UT = {UT_days:.4f} días  |  1 AU/UT = {vel_unit*1e3:.4f} m/s  = {vel_unit:.4f} km/s')
print(f'r_parking = {r_parking_km:.1f} km')
print(f'v_circ(LEO) = {v_circ_km_s:.4f} km/s')
print(f'v_esc (LEO) = {v_esc_km_s:.4f} km/s')

## 5. Efemérides: Apophis en llegada y Tierra en la ventana de lanzamiento

Fijamos la **fecha de llegada** al encuentro histórico: **2029-04-13** (máximo acercamiento de Apophis).
Consultamos su estado heliocéntrico vía NASA Horizons.

Para la ventana de lanzamiento barremos $\tau \in [100, 700]$ días antes de la llegada,
es decir fechas desde ~ agosto 2027 hasta enero 2029.

In [ ]:
# ── Fecha de llegada (encuentro Apophis) ─────────────────────────────────────
FECHA_LLEGADA = '2029-04-13'

print(f'Consultando Apophis en fecha de llegada: {FECHA_LLEGADA} ...')
_, jd_llegada, estado_apo = pc.consulta_horizons(
    id       = '99942',
    location = '@0',
    epochs   = FECHA_LLEGADA,
)

# Posición (AU) y velocidad (AU/UT) de Apophis en la llegada
r_apo = estado_apo[:3]                # AU
v_apo = estado_apo[3:] * UT_days      # AU/día → AU/UT

print(f'  r_Apophis = [{r_apo[0]:.5f}, {r_apo[1]:.5f}, {r_apo[2]:.5f}] AU')
print(f'  v_Apophis = [{v_apo[0]:.5f}, {v_apo[1]:.5f}, {v_apo[2]:.5f}] AU/UT')
print(f'  |r_Apophis| = {np.linalg.norm(r_apo):.5f} AU')
print(f'  |v_Apophis| = {np.linalg.norm(v_apo)*vel_unit:.4f} km/s')

In [ ]:
# ── Ventana de lanzamiento: τ = días antes de la llegada ─────────────────────
# Barremos τ de 100 a 700 días en pasos de 10 días → 61 puntos
tau_arr = np.arange(100, 710, 10)   # días de vuelo

fecha_llegada_dt = datetime.strptime(FECHA_LLEGADA, '%Y-%m-%d')
fechas_lanzamiento = [
    (fecha_llegada_dt - timedelta(days=int(tau))).strftime('%Y-%m-%d')
    for tau in tau_arr
]

print(f'Ventana de lanzamiento: {fechas_lanzamiento[-1]} → {fechas_lanzamiento[0]}')
print(f'τ mínimo: {tau_arr[0]} días  |  τ máximo: {tau_arr[-1]} días')
print(f'Número de fechas a consultar: {len(tau_arr)}')

# Consultar estado heliocéntrico de la Tierra en cada fecha de lanzamiento
# (Una consulta por fecha — la API de Horizons devuelve estado individual para un único epoch)
print('\nConsultando Tierra en cada fecha de lanzamiento (puede tomar ~1 min)...')
estados_tierra = {}
for i, (tau, fecha) in enumerate(zip(tau_arr, fechas_lanzamiento)):
    _, _, estado_t = pc.consulta_horizons(
        id='399', location='@0', epochs=fecha
    )
    estados_tierra[tau] = {
        'fecha': fecha,
        'r': estado_t[:3],                   # AU
        'v': estado_t[3:] * UT_days,         # AU/UT
    }
    if (i + 1) % 10 == 0:
        print(f'  {i+1}/{len(tau_arr)} fechas consultadas...')

print(f'\n✓ Efemérides de la Tierra cargadas para {len(estados_tierra)} fechas.')

## 6. Ejemplo de transferencia: τ = 300 días

Antes de barrer toda la ventana, analizamos en detalle una transferencia con **τ = 300 días**
para ilustrar los cálculos de cada fase.

In [ ]:
tau_ejemplo = 300   # días de vuelo

d_tierra = estados_tierra[tau_ejemplo]
r1 = d_tierra['r']   # posición Tierra (AU)
v1 = d_tierra['v']   # velocidad Tierra (AU/UT)

r2 = r_apo           # posición Apophis en llegada (AU)
v2 = v_apo           # velocidad Apophis en llegada (AU/UT)

tf_UT = tau_ejemplo / UT_days   # tiempo de vuelo en UT

print(f'τ = {tau_ejemplo} días  =  {tf_UT:.4f} UT')
print(f'Fecha lanzamiento: {d_tierra["fecha"]}  →  Llegada: {FECHA_LLEGADA}')
print(f'|r₁| = {np.linalg.norm(r1):.5f} AU  (Tierra al salir)')
print(f'|r₂| = {np.linalg.norm(r2):.5f} AU  (Apophis al llegar)')

# ── Resolver Lambert (solución prograde) ────────────────────────────────────
V1_trans, V2_trans, info_lambert = pc.solucion_lambert(
    r1, r2, tf_UT, mu=mu_sun, direccion='pro'
)

print(f'\n[Lambert prograde]')
print(f'  V₁_trans = {np.linalg.norm(V1_trans)*vel_unit:.4f} km/s  (salida)')
print(f'  V₂_trans = {np.linalg.norm(V2_trans)*vel_unit:.4f} km/s  (llegada)')

In [ ]:
# ── Fase 1: Exceso hiperbólico en salida y ΔV desde LEO ─────────────────────
dv_dep_vec = V1_trans - v1                          # vector exceso respecto a Tierra
v_inf1     = np.linalg.norm(dv_dep_vec) * vel_unit  # km/s

v_p_km_s   = np.sqrt(v_inf1**2 + v_esc_km_s**2)    # vis-viva en perigeo de hipérbola
dv1_km_s   = v_p_km_s - v_circ_km_s

print('=== FASE 1: Escape geocéntrico ===')
print(f'  v_∞₁ (exceso hiperbólico) = {v_inf1:.4f} km/s')
print(f'  v_p  (perigeo hipérbola)  = {v_p_km_s:.4f} km/s')
print(f'  v_circ (LEO 200 km)       = {v_circ_km_s:.4f} km/s')
print(f'  ΔV₁                       = {dv1_km_s:.4f} km/s')

# ── Fase 3: Exceso hiperbólico en llegada (Apophis) ─────────────────────────
dv_arr_vec = V2_trans - v2                          # vector exceso respecto a Apophis
v_inf2     = np.linalg.norm(dv_arr_vec) * vel_unit  # km/s

print('\n=== FASE 3: Llegada a Apophis ===')
print(f'  v_∞₂ (exceso hiperbólico) = {v_inf2:.4f} km/s')
print(f'  ΔV₂ (impactador)         = 0 km/s  (no se frena)')
print(f'  ΔV₂ (rendezvous)         = {v_inf2:.4f} km/s  (iguala velocidad Apophis)')

# ── ΔV total ────────────────────────────────────────────────────────────────
dv_impactador  = dv1_km_s
dv_rendezvous  = dv1_km_s + v_inf2

print(f'\n=== ΔV TOTAL (τ = {tau_ejemplo} días) ===')
print(f'  Impactador  : {dv_impactador:.4f} km/s')
print(f'  Rendezvous  : {dv_rendezvous:.4f} km/s')

# ── Elementos orbitales de la transferencia ─────────────────────────────────
estado_trans = np.concatenate([r1, V1_trans])
elems_trans  = pc.estado_a_elementos(mu_sun, estado_trans)
p_t, e_t, i_t, Om_t, om_t, f_t = elems_trans
a_t = p_t / (1 - e_t**2)

print(f'\n=== Elipse de transferencia heliocéntrica ===')
print(f'  a = {a_t:.4f} AU  |  e = {e_t:.4f}  |  i = {np.degrees(i_t):.2f}°')
print(f'  Tipo: {"elipse" if a_t > 0 else "hipérbola"}')

In [ ]:
# ── Visualización de la trayectoria de la transferencia en el plano eclíptico ─
fig, ax = plt.subplots(figsize=(8, 8))

# Órbita de la Tierra (círculo unitario)
theta = np.linspace(0, 2 * np.pi, 360)
ax.plot(np.cos(theta), np.sin(theta), '--', color='steelblue', linewidth=0.9,
        alpha=0.6, label='Órbita Tierra')

# Órbita de Apophis (kepleriana con elementos pre-encuentro)
_, _, estado_apo_pre = pc.consulta_horizons(id='99942', location='@0', epochs='2028-01-01')
r_pre = estado_apo_pre[:3]
v_pre = estado_apo_pre[3:] * UT_days
elems_apo = pc.estado_a_elementos(mu_sun, np.concatenate([r_pre, v_pre]))
N_f = 500
f_vals = np.linspace(0, 2 * np.pi, N_f)
pos_apo_orb = np.array([
    pc.elementos_a_estado(mu_sun, np.array([elems_apo[0], elems_apo[1],
                                            elems_apo[2], elems_apo[3],
                                            elems_apo[4], fv]))[:3]
    for fv in f_vals
])
ax.plot(pos_apo_orb[:, 0], pos_apo_orb[:, 1], '-', color='tomato', linewidth=1.0,
        alpha=0.7, label='Órbita Apophis')

# Arco de transferencia (integración de dos cuerpos)
ts_trans = np.linspace(0, tf_UT, 300)
sistema_trans = [
    dict(m=mu_sun, r=[0, 0, 0], v=[0, 0, 0]),
    dict(m=1e-20,  r=list(r1),  v=list(V1_trans)),
]
rs_t, _, _, _, _ = pc.ncuerpos_solucion(sistema_trans, ts_trans)
tray = rs_t[1, :, :]  # trayectoria de la nave
ax.plot(tray[:, 0], tray[:, 1], '-', color='darkorange', linewidth=2.0,
        label=f'Transferencia (τ={tau_ejemplo} d, e={e_t:.3f})')

# Marcadores: Sol, Tierra al lanzar, Apophis al llegar
ax.plot(0, 0, 'yo', markersize=14, zorder=5, label='Sol')
ax.plot(r1[0], r1[1], 'bs', markersize=9, zorder=5, label=f'Tierra ({d_tierra["fecha"]})')
ax.plot(r2[0], r2[1], 'r^', markersize=9, zorder=5, label=f'Apophis ({FECHA_LLEGADA})')

# Vectores de impulso
escala = 0.08
dv1_vec_AU_UT = dv_dep_vec
dv1_norm = dv1_vec_AU_UT / np.linalg.norm(dv1_vec_AU_UT)
ax.annotate('', xy=(r1[0] + escala*dv1_norm[0], r1[1] + escala*dv1_norm[1]),
            xytext=(r1[0], r1[1]),
            arrowprops=dict(arrowstyle='->', color='blue', lw=2.5))

ax.set_aspect('equal')
ax.set_xlabel('$x$ (AU)', fontsize=12)
ax.set_ylabel('$y$ (AU)', fontsize=12)
ax.set_title(f'Transferencia heliocéntrica Tierra→Apophis (τ = {tau_ejemplo} días)\n'
             f'ΔV₁ = {dv1_km_s:.3f} km/s  |  v_∞₂ = {v_inf2:.3f} km/s', fontsize=12)
ax.legend(fontsize=9, loc='upper left')
ax.grid(alpha=0.25)

plt.tight_layout()
plt.savefig('transferencia_ejemplo.png', dpi=130, bbox_inches='tight')
plt.show()
print('Figura guardada: transferencia_ejemplo.png')

## 7. Barrido ΔV(τ): ventana completa de lanzamiento

Iteramos sobre todos los τ de la ventana, resolviendo Lambert en cada caso.
Se calculan las ΔV para **dos estrategias de misión**:

- **Impactador** (DART-like): $\Delta V = \Delta V_1$ solamente.
- **Rendezvous**: $\Delta V = \Delta V_1 + v_{\infty,2}$.

In [ ]:
def calcular_dv_transferencia(r1, v1, r2, v2, tau_dias, mu=1.0):
    """Calcula ΔV₁, v_∞₂ y ΔV_total para una transferencia Tierra→Apophis.

    Parámetros
    ----------
    r1, v1 : ndarray (3,)  — posición y velocidad de la Tierra al lanzar [AU, AU/UT]
    r2, v2 : ndarray (3,)  — posición y velocidad de Apophis al llegar   [AU, AU/UT]
    tau_dias : float       — tiempo de vuelo [días]
    mu      : float        — parámetro gravitacional del Sol [canónico]

    Retorna
    -------
    dict con 'dv1', 'v_inf2', 'dv_impactador', 'dv_rendezvous',
             'v_inf1', 'a_trans', 'e_trans' — todos en km/s o AU según se indica.
    Retorna None si Lambert no converge.
    """
    tf_UT = tau_dias / UT_days
    try:
        V1t, V2t, _ = pc.solucion_lambert(r1, r2, tf_UT, mu=mu, direccion='pro')
    except Exception:
        return None

    # Excesos hiperbólicos
    v_inf1 = np.linalg.norm(V1t - v1) * vel_unit    # km/s
    v_inf2 = np.linalg.norm(V2t - v2) * vel_unit    # km/s

    # ΔV₁ desde LEO (vis-viva en perigeo de hipérbola de escape)
    v_p  = np.sqrt(v_inf1**2 + v_esc_km_s**2)
    dv1  = v_p - v_circ_km_s

    # Elementos de la transferencia
    try:
        elems = pc.estado_a_elementos(mu, np.concatenate([r1, V1t]))
        p_t, e_t = elems[0], elems[1]
        a_t = p_t / (1 - e_t**2)
    except Exception:
        a_t, e_t = np.nan, np.nan

    return {
        'v_inf1'        : v_inf1,
        'v_inf2'        : v_inf2,
        'dv1'           : dv1,
        'dv_impactador' : dv1,
        'dv_rendezvous' : dv1 + v_inf2,
        'a_trans'       : a_t,
        'e_trans'       : e_t,
    }


# ── Ejecutar barrido ─────────────────────────────────────────────────────────
print('Calculando ΔV para cada τ...')
resultados = []

for tau in tau_arr:
    d_t = estados_tierra[tau]
    res = calcular_dv_transferencia(
        d_t['r'], d_t['v'], r_apo, v_apo, tau
    )
    if res is not None:
        res['tau']         = tau
        res['fecha_lanzo'] = d_t['fecha']
        resultados.append(res)

df = pd.DataFrame(resultados)
print(f'\n✓ Calculados {len(df)} puntos de la curva ΔV(τ).')
print(df[['tau', 'fecha_lanzo', 'dv1', 'v_inf2', 'dv_impactador', 'dv_rendezvous']].to_string(index=False))

## 8. Visualización: curvas ΔV(τ) y ventana óptima

### 8.1 Curva ΔV(τ)

Graficamos las tres componentes como función del tiempo de vuelo $\tau$:
- **$\Delta V_1$**: costo de salida desde LEO
- **$v_{\infty,2}$**: exceso de velocidad en llegada
- **$\Delta V_{\rm total}$ (rendezvous)**: suma de ambas

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 10), sharex=True)

tau_vals  = df['tau'].values
dv1_vals  = df['dv1'].values
vinf2_vals = df['v_inf2'].values
dv_imp    = df['dv_impactador'].values
dv_rdz    = df['dv_rendezvous'].values

# ─── Panel superior: componentes individuales ───────────────────────────────
ax = axes[0]
ax.plot(tau_vals, dv1_vals,   '-o', color='steelblue',  markersize=3, linewidth=1.8,
        label=r'$\Delta V_1$ (salida LEO)')
ax.plot(tau_vals, vinf2_vals, '-s', color='tomato',     markersize=3, linewidth=1.8,
        label=r'$v_{\infty,2}$ (llegada Apophis)')

# Mínimo de ΔV₁
idx_min_dv1 = np.argmin(dv1_vals)
ax.axvline(tau_vals[idx_min_dv1], color='steelblue', linestyle=':', alpha=0.7)
ax.plot(tau_vals[idx_min_dv1], dv1_vals[idx_min_dv1], 'b*', markersize=14,
        label=f'min ΔV₁ = {dv1_vals[idx_min_dv1]:.3f} km/s @ τ={tau_vals[idx_min_dv1]:.0f} d')

ax.set_ylabel('ΔV (km/s)', fontsize=12)
ax.set_title('Ventana de lanzamiento DART inverso — Tierra → Apophis 2029-04-13', fontsize=13)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
ax.yaxis.set_minor_locator(mticker.AutoMinorLocator())

# ─── Panel inferior: ΔV total (impactador vs. rendezvous) ───────────────────
ax2 = axes[1]
ax2.plot(tau_vals, dv_imp, '-', color='darkorange',  linewidth=2.2,
         label=r'$\Delta V_{\rm total}$ impactador  ($\Delta V_1$ solo)')
ax2.plot(tau_vals, dv_rdz, '--', color='darkviolet', linewidth=2.2,
         label=r'$\Delta V_{\rm total}$ rendezvous  ($\Delta V_1 + v_{\infty,2}$)')

# Mínimo rendezvous
idx_min_rdz = np.argmin(dv_rdz)
ax2.axvline(tau_vals[idx_min_rdz], color='darkviolet', linestyle=':', alpha=0.7)
ax2.plot(tau_vals[idx_min_rdz], dv_rdz[idx_min_rdz], 'v', color='darkviolet',
         markersize=12, zorder=5,
         label=f'min rendez. = {dv_rdz[idx_min_rdz]:.3f} km/s @ τ={tau_vals[idx_min_rdz]:.0f} d')

# Mínimo impactador
idx_min_imp = np.argmin(dv_imp)
ax2.plot(tau_vals[idx_min_imp], dv_imp[idx_min_imp], '^', color='darkorange',
         markersize=12, zorder=5,
         label=f'min impact. = {dv_imp[idx_min_imp]:.3f} km/s @ τ={tau_vals[idx_min_imp]:.0f} d')

ax2.set_xlabel('Tiempo de vuelo τ (días antes del encuentro)', fontsize=12)
ax2.set_ylabel('ΔV total (km/s)', fontsize=12)
ax2.legend(fontsize=10)
ax2.grid(alpha=0.3)
ax2.yaxis.set_minor_locator(mticker.AutoMinorLocator())

# Ejes secundarios con fecha de lanzamiento
ax2_top = ax2.twiny()
ax2_top.set_xlim(ax2.get_xlim())
tau_ticks = np.arange(100, 710, 100)
ax2_top.set_xticks(tau_ticks)
ax2_top.set_xticklabels(
    [(fecha_llegada_dt - timedelta(days=int(t))).strftime('%Y-%m') for t in tau_ticks],
    fontsize=8, rotation=20
)
ax2_top.set_xlabel('Fecha de lanzamiento aproximada', fontsize=10)

plt.tight_layout()
plt.savefig('dv_ventana_lanzamiento.png', dpi=130, bbox_inches='tight')
plt.show()
print('Figura guardada: dv_ventana_lanzamiento.png')

In [ ]:
# ── 8.2 Mapa de elementos de la transferencia en función de τ ───────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

a_vals = df['a_trans'].values
e_vals = df['e_trans'].values

# Panel izquierdo: semiejor mayor
ax = axes[0]
ax.plot(tau_vals, a_vals, '-o', color='teal', markersize=4, linewidth=1.8)
ax.axhline(1.0, color='steelblue', linestyle='--', linewidth=1, label='Tierra (1 AU)')
ax.axhline(np.linalg.norm(r_apo), color='tomato', linestyle='--', linewidth=1,
           label=f'Apophis ({np.linalg.norm(r_apo):.3f} AU al llegar)')
ax.set_xlabel('τ (días)', fontsize=11)
ax.set_ylabel('a de la transferencia (AU)', fontsize=11)
ax.set_title('Semiejor mayor de la elipse de transferencia', fontsize=11)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# Panel derecho: excentricidad
ax = axes[1]
ax.plot(tau_vals, e_vals, '-o', color='firebrick', markersize=4, linewidth=1.8)
ax.axhline(1.0, color='gray', linestyle=':', linewidth=1, label='e=1 (parábola)')
ax.set_xlabel('τ (días)', fontsize=11)
ax.set_ylabel('Excentricidad e', fontsize=11)
ax.set_title('Excentricidad de la elipse de transferencia', fontsize=11)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

plt.suptitle('Geometría de la transferencia heliocéntrica vs. τ', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('geometria_transferencia.png', dpi=130, bbox_inches='tight')
plt.show()
print('Figura guardada: geometria_transferencia.png')

In [ ]:
# ── 8.3 Resumen: tabla de la ventana óptima ──────────────────────────────────
print('=' * 80)
print('VENTANA ÓPTIMA DE LANZAMIENTO — DART INVERSO (Tierra → Apophis 2029-04-13)')
print('=' * 80)

# Filtrar τ donde ΔV_rendezvous < cuantil 30%
umbral_rdz = np.percentile(dv_rdz, 30)
mascara    = dv_rdz <= umbral_rdz

df_ventana = df[mascara][['tau', 'fecha_lanzo', 'dv1', 'v_inf2',
                           'dv_impactador', 'dv_rendezvous',
                           'a_trans', 'e_trans']].copy()
df_ventana.columns = ['τ (d)', 'Fecha lanzam.', 'ΔV₁ (km/s)', 'v∞₂ (km/s)',
                      'ΔV impactador', 'ΔV rendezvous', 'a (AU)', 'e']

pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_rows', None)
print(df_ventana.to_string(index=False))

# ── Óptimo global ──────────────────────────────────────────────────────────
optimo_imp = df.loc[df['dv_impactador'].idxmin()]
optimo_rdz = df.loc[df['dv_rendezvous'].idxmin()]

print('\n' + '-' * 80)
print('ÓPTIMOS GLOBALES:')
print(f'  Impactador:  τ = {optimo_imp["tau"]:.0f} días  |  '
      f'lanzam. {optimo_imp["fecha_lanzo"]}  |  '
      f'ΔV₁ = {optimo_imp["dv1"]:.4f} km/s')
print(f'  Rendezvous:  τ = {optimo_rdz["tau"]:.0f} días  |  '
      f'lanzam. {optimo_rdz["fecha_lanzo"]}  |  '
      f'ΔV_total = {optimo_rdz["dv_rendezvous"]:.4f} km/s  '
      f'(ΔV₁={optimo_rdz["dv1"]:.3f}  +  v∞₂={optimo_rdz["v_inf2"]:.3f} km/s)')
print('=' * 80)

## 9. Conclusiones

### Resultados obtenidos

El modelo de **parches cónicos** para la misión DART inverso (Tierra → Apophis, llegada 2029-04-13) revela:

1. **Ventana de mínimo costo** para el impactador y el rendezvous se ubica en un rango de ~τ óptimo días
   antes del encuentro (los valores exactos dependen del resultado del barrido).

2. **Curva ΔV(τ) no monótona:** para tiempos de vuelo cortos (τ < 200 d) el costo sube porque la
   geometría de la transferencia requiere velocidades relativas más altas. Para τ muy largos (> 600 d)
   también aumenta porque la posición de la Tierra se aleja de la geometría favorable.

3. **Ecuación vis-viva y exceso hiperbólico:** el impulso $\Delta V_1$ desde LEO es sensible a $v_{\infty,1}$;
   una pequeña reducción de $v_{\infty,1}$ conlleva una reducción apreciable de $\Delta V_1$ gracias al
   **efecto Oberth** (la órbita baja amplifica el impulso).

### Limitaciones del modelo

| Limitación | Impacto estimado |
|---|---|
| Se asume transfer en el plano de la eclíptica (2D aprox.) | Subestima el $\Delta V$ de cambio de plano |
| No se modela el $\Delta V$ de inyección en LEO | Error < 0.1 km/s en condiciones nominales |
| Lambert con llegada exacta a posición Apophis | No considera dispersión de guiado |
| Apophis tratado como punto (sin gravedad) | Válido para impactador; relevante para órbita apofocéntrica |

### Siguientes pasos

- **NB01:** Optimización de la transferencia con cambio de plano (inclination correction).
- **NB02:** Análisis de incertidumbre — propagación de errores de efemérides a ΔV.
- **NB03:** Integración N-cuerpos completa de la trayectoria de la nave para validar el modelo de parches cónicos.